In [1]:
from pathlib import Path
from datetime import datetime, timezone
from io import BytesIO
import sys
import time
import json
import numpy as np
import pandas as pd
import requests
from IPython.display import display

SEASONS = [2021, 2022, 2023, 2024, 2025]
SPLIT = {2021:'TRAIN', 2022:'TRAIN', 2023:'TRAIN', 2024:'VALIDATION', 2025:'TEST'}
SCHEDULE_URL = 'https://raw.githubusercontent.com/nflverse/nfldata/master/data/games.csv'
TEAM_STATS_URL = 'https://github.com/nflverse/nflverse-data/releases/download/stats_team/stats_team_week_{season}.csv'

current = Path.cwd().resolve()
if current.name.lower().replace('_','-') in {'against-all-odds','againstallodds'}:
    ROOT = current
elif 'google.colab' in sys.modules:
    ROOT = Path('/content/againstallodds')
else:
    ROOT = current / 'againstallodds'
RAW = ROOT/'data'/'raw'/'nfl'
CLEAN = ROOT/'data'/'clean'/'nfl'
DOCS = ROOT/'docs'
RESULTS = ROOT/'results'
for folder in (RAW,CLEAN,DOCS,RESULTS):
    folder.mkdir(parents=True,exist_ok=True)
print('Project root:', ROOT)

# Use a cached copy for repeatable notebook reruns.
session = requests.Session()
session.headers.update({'User-Agent':'AgainstAllOdds-Capstone/1.0'})

def get_csv(url, cache_path, *, optional=False):
    if cache_path.exists() and cache_path.stat().st_size > 0:
        print('Cached:', cache_path.name)
        return pd.read_csv(cache_path, low_memory=False, dtype={'game_id':'string'}), None
    error = None
    for attempt in range(3):
        try:
            r = session.get(url, timeout=90)
            r.raise_for_status()
            candidate = pd.read_csv(BytesIO(r.content), low_memory=False, dtype={'game_id':'string'})
            if candidate.empty:
                raise ValueError('The source returned an empty CSV')
            cache_path.write_bytes(r.content)
            print('Downloaded:', cache_path.name, '- rows:',len(candidate))
            return candidate, None
        except (requests.RequestException, ValueError, pd.errors.ParserError) as exc:
            error = str(exc)
            if attempt < 2: time.sleep(2*(attempt+1))
    if optional:
        print('Optional source unavailable:', url, error)
        return pd.DataFrame(), error
    raise RuntimeError(f'Required source could not be downloaded: {url}; {error}')

raw, _ = get_csv(SCHEDULE_URL, RAW/'nflverse_games_source.csv')
print('Raw schedule shape:',raw.shape)
display(raw.head(5))

# Verify the source schema rather than silently inventing columns.
required = ['game_id','season','week','game_type','gameday','home_team','away_team','home_score','away_score']
missing = sorted(set(required)-set(raw.columns))
if missing:
    raise ValueError(f'Schedule schema changed. Missing fields: {missing}. Inspect the source before proceeding.')

NFL_NAMES = {
'ARI':'Arizona Cardinals','ATL':'Atlanta Falcons','BAL':'Baltimore Ravens','BUF':'Buffalo Bills',
'CAR':'Carolina Panthers','CHI':'Chicago Bears','CIN':'Cincinnati Bengals','CLE':'Cleveland Browns',
'DAL':'Dallas Cowboys','DEN':'Denver Broncos','DET':'Detroit Lions','GB':'Green Bay Packers',
'HOU':'Houston Texans','IND':'Indianapolis Colts','JAX':'Jacksonville Jaguars','KC':'Kansas City Chiefs',
'LV':'Las Vegas Raiders','LAC':'Los Angeles Chargers','LA':'Los Angeles Rams','MIA':'Miami Dolphins',
'MIN':'Minnesota Vikings','NE':'New England Patriots','NO':'New Orleans Saints','NYG':'New York Giants',
'NYJ':'New York Jets','PHI':'Philadelphia Eagles','PIT':'Pittsburgh Steelers','SEA':'Seattle Seahawks',
'SF':'San Francisco 49ers','TB':'Tampa Bay Buccaneers','TEN':'Tennessee Titans','WAS':'Washington Commanders'}
assert len(NFL_NAMES)==32
# Historical aliases. Do not overwrite a team's source label without documenting it.
ALIASES={'STL':'LA','SD':'LAC','OAK':'LV','JAC':'JAX','WSH':'WAS'}

def standardize(series):
    return series.astype('string').str.strip().str.upper().replace(ALIASES)

nfl = raw.loc[pd.to_numeric(raw['season'],errors='coerce').isin(SEASONS)].copy()
nfl = nfl.loc[nfl['game_type'].astype('string').str.upper().eq('REG')].copy()
nfl['season'] = pd.to_numeric(nfl['season'],errors='coerce').astype('Int64')
nfl['week'] = pd.to_numeric(nfl['week'],errors='coerce').astype('Int64')
nfl['game_id'] = nfl['game_id'].astype('string').str.strip()
nfl['game_date'] = pd.to_datetime(nfl['gameday'],errors='coerce').dt.normalize()
nfl['home_team_source'] = nfl['home_team']
nfl['away_team_source'] = nfl['away_team']
nfl['home_team'] = standardize(nfl['home_team'])
nfl['away_team'] = standardize(nfl['away_team'])
nfl['home_team_name'] = nfl['home_team'].map(NFL_NAMES)
nfl['away_team_name'] = nfl['away_team'].map(NFL_NAMES)
nfl['home_score'] = pd.to_numeric(nfl['home_score'],errors='coerce')
nfl['away_score'] = pd.to_numeric(nfl['away_score'],errors='coerce')
nfl['data_split'] = nfl['season'].map(SPLIT)
print('Regular-season rows selected:',len(nfl))
print('Unexpected team codes:', sorted((set(nfl['home_team'].dropna())|set(nfl['away_team'].dropna()))-set(NFL_NAMES)))

# Retain issue counts instead of quietly deleting data.
critical = ['game_id','season','week','game_date','home_team_name','away_team_name','home_score','away_score']
raw_missing = pd.DataFrame({'field':nfl.columns,'missing_count':nfl.isna().sum().values,
                            'missing_percent':(100*nfl.isna().mean()).round(2).values})
display(raw_missing.sort_values('missing_percent',ascending=False).head(20))
invalid_mask = nfl[critical].isna().any(axis=1) | nfl['home_team'].eq(nfl['away_team']).fillna(True)
invalid = nfl.loc[invalid_mask].copy()
print('Rows with missing critical data / invalid teams:',len(invalid))
if len(invalid): display(invalid[['game_id','season','week','home_team','away_team','home_score','away_score']].head(15))
# The modeling dataset must contain only final, usable records.
nfl = nfl.loc[~invalid_mask].copy()
prior_dupes = int(nfl['game_id'].duplicated().sum())
nfl = nfl.drop_duplicates('game_id',keep='first').copy()
print('Duplicate game IDs removed:',prior_dupes)

# A tie has a genuine result but no winner; keep it in master data and mark binary label missing.
nfl['HOME_MARGIN'] = nfl['home_score'] - nfl['away_score']
nfl['result_category'] = np.select([nfl['HOME_MARGIN'].gt(0),nfl['HOME_MARGIN'].lt(0)],
                                    ['HOME_WIN','AWAY_WIN'],default='TIE')
nfl['HOME_WIN'] = pd.Series(np.where(nfl['HOME_MARGIN'].eq(0), pd.NA,
                            np.where(nfl['HOME_MARGIN'].gt(0),1,0)), index=nfl.index).astype('Int64')
nfl['is_tie'] = nfl['HOME_MARGIN'].eq(0)
nfl['home_is_home'] = 1
nfl['away_is_home'] = 0
nfl['is_neutral_site'] = (nfl['location'].astype('string').str.lower().eq('neutral')
                          if 'location' in nfl.columns else pd.Series(pd.NA,index=nfl.index,dtype='boolean'))
nfl = nfl.sort_values(['game_date','week','game_id'],kind='stable').reset_index(drop=True)

selected=['game_id','season','week','game_date','data_split','home_team','away_team',
          'home_team_name','away_team_name','home_score','away_score','HOME_MARGIN',
          'result_category','HOME_WIN','is_tie','home_is_home','away_is_home','is_neutral_site']
optional_cols=[c for c in ['gametime','weekday','stadium','roof','surface','home_rest','away_rest'] if c in nfl]
games=nfl[selected+optional_cols].copy()
model_games=games.loc[~games['is_tie']].copy()
assert model_games['HOME_WIN'].notna().all()
print('Master games:',len(games),'| ties:',int(games.is_tie.sum()),'| binary modeling games:',len(model_games))
display(games.head())

# One game produces exactly two team-game observations.
home = games[['game_id','season','week','game_date','data_split','home_team','away_team','home_score','away_score','HOME_WIN','is_tie']].rename(
 columns={'home_team':'team_abbr','away_team':'opponent_abbr','home_score':'points_for','away_score':'points_against'})
home['is_home']=1
away = games[['game_id','season','week','game_date','data_split','home_team','away_team','home_score','away_score','HOME_WIN','is_tie']].rename(
 columns={'away_team':'team_abbr','home_team':'opponent_abbr','away_score':'points_for','home_score':'points_against'})
away['is_home']=0
team_games=pd.concat([home,away],ignore_index=True).sort_values(
 ['game_date','game_id','is_home'],ascending=[True,True,False],kind='stable').reset_index(drop=True)
team_games['team_name']=team_games['team_abbr'].map(NFL_NAMES)
team_games['opponent_name']=team_games['opponent_abbr'].map(NFL_NAMES)
team_games['point_differential']=team_games['points_for']-team_games['points_against']
team_games['team_result']=np.select([team_games['point_differential'].gt(0),team_games['point_differential'].lt(0)],
 ['W','L'],default='T')
assert len(team_games)==2*len(games)
assert team_games.groupby('game_id')['is_home'].agg(['sum','count']).eq({'sum':1,'count':2}).all().all()
display(team_games.head(6))

stat_attempts=[]
stat_parts=[]
for season in SEASONS:
    url=TEAM_STATS_URL.format(season=season)
    frame,error=get_csv(url,RAW/f'stats_team_week_{season}.csv',optional=True)
    stat_attempts.append({'season':season,'download_ok':error is None,'rows':len(frame),'source':url,'error':error or ''})
    if not frame.empty:
        frame['requested_season']=season
        stat_parts.append(frame)
stat_audit=pd.DataFrame(stat_attempts)
display(stat_audit)
team_week_stats=pd.concat(stat_parts,ignore_index=True) if stat_parts else pd.DataFrame()
print('Extra team-week stat rows:',len(team_week_stats))
if not team_week_stats.empty:
    print('Extra stat columns:', team_week_stats.columns.tolist())
    display(team_week_stats.head())

# Build a quick statistical summary from *completed game results* for all teams.
team_season=(team_games.groupby(['season','data_split','team_abbr','team_name'],as_index=False)
 .agg(games=('game_id','nunique'),wins=('team_result',lambda v:int((v=='W').sum())),
      losses=('team_result',lambda v:int((v=='L').sum())),ties=('team_result',lambda v:int((v=='T').sum())),
      avg_points_for=('points_for','mean'),avg_points_against=('points_against','mean'),
      avg_point_differential=('point_differential','mean')))
team_season['win_pct']=(team_season['wins']+0.5*team_season['ties'])/team_season['games']
# IMPORTANT: full-season summaries are descriptive outcomes and must NOT be joined to each game as predictors.
coverage=games.groupby(['season','data_split'],as_index=False).agg(
 games=('game_id','nunique'),first_game=('game_date','min'),last_game=('game_date','max'),ties=('is_tie','sum'))
coverage['unique_teams']=[len(set(games.loc[games['season'].eq(s),'home_team']) |
                                set(games.loc[games['season'].eq(s),'away_team'])) for s in coverage['season']]
coverage['expected_games']=272
coverage['coverage_ok']=coverage['games'].eq(272)&coverage['unique_teams'].eq(32)
display(coverage)

# Fail loudly on critical integrity errors; report source coverage separately.
checks={
 'five_seasons': games.season.nunique()==5,
 'unique_game_id': games.game_id.is_unique,
 'all_32_teams_each_season': coverage.unique_teams.eq(32).all(),
 '272_regular_season_games_per_season': coverage.games.eq(272).all(),
 'chronological_order': games.game_date.is_monotonic_increasing,
 'no_missing_critical_fields': not games[['game_id','game_date','home_team','away_team','home_score','away_score']].isna().any().any(),
 'two_team_rows_per_game': len(team_games)==2*len(games),
 'ties_have_missing_binary_label': games.loc[games.is_tie,'HOME_WIN'].isna().all(),
 'non_ties_have_binary_label': games.loc[~games.is_tie,'HOME_WIN'].isin([0,1]).all(),
 'correct_score_based_winner': games.loc[~games.is_tie,'HOME_WIN'].eq(games.loc[~games.is_tie,'HOME_MARGIN'].gt(0).astype(int)).all(),
}
qc=pd.DataFrame([{'check':k,'passed':bool(v)} for k,v in checks.items()]);display(qc)

# Save generated data and documentation locally in the Colab runtime.
for frame,path in [
 (games,CLEAN/'nfl_games_clean_2021_to_2025.csv'),
 (model_games,CLEAN/'nfl_games_binary_no_ties_2021_to_2025.csv'),
 (team_games,CLEAN/'nfl_team_game_logs_clean_2021_to_2025.csv'),
 (team_season,CLEAN/'nfl_team_season_summary_2021_to_2025.csv'),
 (coverage,RESULTS/'week4_nfl_coverage.csv'),
 (qc,RESULTS/'week4_nfl_quality_checks.csv'),
 (stat_audit,DOCS/'week4_nfl_extra_stats_source_audit.csv'),
 (raw_missing,DOCS/'week4_nfl_source_missingness.csv'),
]:
    frame.to_csv(path,index=False)
    print('Saved:',path)

# Separate dataset schema from methodological warnings.
dict_rows=[
 ('game_id','source game identifier','pre-game identifier'),
 ('season','NFL season starting year','pre-game identifier'),
 ('week','regular-season week','pre-game identifier'),
 ('game_date','scheduled date, normalized','pre-game context'),
 ('home_team','standardized home abbreviation','pre-game context'),
 ('away_team','standardized away abbreviation','pre-game context'),
 ('home_score','home final points','post-game outcome'),
 ('away_score','away final points','post-game outcome'),
 ('HOME_MARGIN','home_score minus away_score','post-game outcome'),
 ('HOME_WIN','1 home win, 0 away win, missing if tied','post-game target'),
 ('is_tie','whether final scores are equal','post-game outcome'),
 ('team_result','W / L / T in team-game row','post-game outcome'),
 ('points_for','team final score','post-game historical stat'),
 ('points_against','opponent final score','post-game historical stat'),
 ('point_differential','points_for minus points_against','post-game historical stat'),
 ('data_split','TRAIN 2021-23; VALIDATION 2024; TEST 2025','study design'),
]
dictionary=pd.DataFrame(dict_rows,columns=['variable_name','description','availability'])
dictionary.insert(1,'sport','NFL')
dictionary.to_csv(DOCS/'week4_nfl_data_dictionary.csv',index=False)

summary=(f'# Week 4 — NFL collection and cleaning\n\n'
 f'Completed regular-season seasons: {SEASONS}.\n\n'
 f'Schedule source: {SCHEDULE_URL}\n\n'
 f'Games retained: {len(games)}; ties: {int(games.is_tie.sum())}; binary modeling games: {len(model_games)}.\n\n'
 f'Critical checks all passed: {bool(qc.passed.all())}.\n\n'
 f'Extra team-stat seasons downloaded: {int(stat_audit.download_ok.sum())}/5. '
 f'If incomplete, only score-derived team-game statistics are guaranteed.\n\n'
 'Pre-game rule: current-game results and detailed statistics are post-game only. '
 'Use previous completed games with lagged features in Week 6.\n')
(DOCS/'week4_nfl_documentation.md').write_text(summary,encoding='utf-8')
print('Week 4:', 'PASS' if qc.passed.all() else 'REVIEW FAILED CHECKS')


Project root: /content/againstallodds
Downloaded: nflverse_games_source.csv - rows: 7548
Raw schedule shape: (7548, 46)


,game_id,season,game_type,week,gameday,weekday,gametime,away_team,away_score,home_team,...,wind,away_qb_id,home_qb_id,away_qb_name,home_qb_name,away_coach,home_coach,referee,stadium_id,stadium
0,1999_01_MIN_ATL,1999,REG,1,1999-09-12,Sunday,NaN,MIN,17.0,ATL,...,NaN,00-0003761,00-0002876,Randall Cunningham,Chris Chandler,Dennis Green,Dan Reeves,Gerry Austin,ATL00,Georgia Dome
1,1999_01_KC_CHI,1999,REG,1,1999-09-12,Sunday,NaN,KC,17.0,CHI,...,12.0,00-0006300,00-0010560,Elvis Grbac,Shane Matthews,Gunther Cunningham,Dick Jauron,Phil Luckett,CHI98,Soldier Field
2,1999_01_PIT_CLE,1999,REG,1,1999-09-12,Sunday,NaN,PIT,43.0,CLE,...,12.0,00-0015700,00-0004230,Kordell Stewart,Ty Detmer,Bill Cowher,Chris Palmer,Bob McElwee,CLE00,Cleveland Browns Stadium
3,1999_01_OAK_GB,1999,REG,1,1999-09-12,Sunday,NaN,OAK,24.0,GB,...,10.0,00-0005741,00-0005106,Rich Gannon,Brett Favre,Jon Gruden,Ray Rhodes,Tony Corrente,GNB00,Lambeau Field
4,1999_01_BUF_IND,1999,REG,1,1999-09-12,Sunday,NaN,BUF,14.0,IND,...,NaN,00-0005363,00-0010346,Doug Flutie,Peyton Manning,Wade Phillips,Jim Mora,Ron Blum,IND99,RCA Dome


Regular-season rows selected: 1359
Unexpected team codes: []


,field,missing_count,missing_percent
17,nfl_detail_id,1087,79.99
19,pff,815,59.97
36,wind,581,42.75
35,temp,581,42.75
21,ftn,272,20.01
34,surface,44,3.24
43,referee,1,0.07
4,gameday,0,0.00
3,week,0,0.00
2,game_type,0,0.00


Rows with missing critical data / invalid teams: 0
Duplicate game IDs removed: 0
Master games: 1359 | ties: 4 | binary modeling games: 1355


...

Extra team-week stat rows: 2848
Extra stat columns: ['season', 'week', 'team', 'season_type', 'game_id', 'opponent_team', 'completions', 'attempts', 'passing_yards', 'passing_tds', 'passing_interceptions', 'sacks_suffered', 'sack_yards_lost', 'sack_fumbles', 'sack_fumbles_lost', 'passing_air_yards', 'passing_yards_after_catch', 'passing_first_downs', 'passing_epa', 'passing_cpoe', 'passing_2pt_conversions', 'passing_10', 'passing_16', 'passing_20', 'passing_40', 'carries', 'rushing_yards', 'rushing_tds', 'rushing_fumbles', 'rushing_fumbles_lost', 'rushing_first_downs', 'rushing_epa', 'rushing_2pt_conversions', 'rushing_10', 'rushing_12', 'rushing_20', 'rushing_40', 'receptions', 'targets', 'receiving_yards', 'receiving_tds', 'receiving_fumbles', 'receiving_fumbles_lost', 'receiving_air_yards', 'receiving_yards_after_catch', 'receiving_first_downs', 'receiving_epa', 'receiving_2pt_conversions', 'receiving_10', 'receiving_16', 'receiving_20', 'receiving_40', 'special_teams_tds', 'def_tac

,season,week,team,season_type,game_id,opponent_team,completions,attempts,passing_yards,passing_tds,...,pt_inside_20,pt_out_of_bounds,pt_downed,pt_touchback,pt_fair_caught,pt_returned,pt_return_yards,pt_return_tds,pt_net_yards,requested_season
0,2021,1,ARI,REG,2021_01_ARI_TEN,TEN,21,32,289,4,...,2,0,1,1,1,0,0,0,124,2021
1,2021,1,ATL,REG,2021_01_PHI_ATL,PHI,21,35,164,0,...,2,1,0,0,1,4,19,0,260,2021
2,2021,1,BAL,REG,2021_01_BAL_LV,LV,19,30,235,1,...,1,0,1,0,1,2,10,0,169,2021
3,2021,1,BUF,REG,2021_01_PIT_BUF,PIT,30,51,270,1,...,2,1,0,1,0,1,5,0,112,2021
4,2021,1,CAR,REG,2021_01_NYJ_CAR,NYJ,24,35,279,1,...,2,1,1,0,3,1,15,0,210,2021


,season,data_split,games,first_game,last_game,ties,unique_teams,expected_games,coverage_ok
0,2021,TRAIN,272,2021-09-09,2022-01-09,1,32,272,True
1,2022,TRAIN,271,2022-09-08,2023-01-08,2,32,272,False
2,2023,TRAIN,272,2023-09-07,2024-01-07,0,32,272,True
3,2024,VALIDATION,272,2024-09-05,2025-01-05,0,32,272,True
4,2025,TEST,272,2025-09-04,2026-01-04,1,32,272,True


,check,passed
0,five_seasons,True
1,unique_game_id,True
2,all_32_teams_each_season,True
3,272_regular_season_games_per_season,False
4,chronological_order,True
5,no_missing_critical_fields,True
6,two_team_rows_per_game,True
7,ties_have_missing_binary_label,True
8,non_ties_have_binary_label,True
9,correct_score_based_winner,True


Saved: /content/againstallodds/data/clean/nfl/nfl_games_clean_2021_to_2025.csv
Saved: /content/againstallodds/data/clean/nfl/nfl_games_binary_no_ties_2021_to_2025.csv
Saved: /content/againstallodds/data/clean/nfl/nfl_team_game_logs_clean_2021_to_2025.csv
Saved: /content/againstallodds/data/clean/nfl/nfl_team_season_summary_2021_to_2025.csv
Saved: /content/againstallodds/results/week4_nfl_coverage.csv
Saved: /content/againstallodds/results/week4_nfl_quality_checks.csv
Saved: /content/againstallodds/docs/week4_nfl_extra_stats_source_audit.csv
Saved: /content/againstallodds/docs/week4_nfl_source_missingness.csv
Week 4: REVIEW FAILED CHECKS
